In [ ]:
from  agents import Agent, Runner, function_tool, ItemHelpers

@function_tool
def get_weather(city: str):
    """get weather by city"""
    print(city)
    return "30 degrees"

agent = Agent(
    name = "Assistant Agent",
    instructions = "You are a helpful assistant. Use tools when needed to answer questions",
    tools = [get_weather]
)

# runner는 while true loop처리하는 것
# runner는 기본적으로 agent랑 input 받음(물론 agent에 tool, sub agent가 있을 수도 있지만)
# runner는 main agent, input 받아서, openai에 request 보내면 openai가 응답하고,
# runner는 그 response를 추출해서 parsing하고, 호출해야 할 tool있는지 확인함
# 확인 후 runner가 해당 tool 호출 후 결과를 openai로 전송함
# runner가 final response를 받아서 우리에게 출력함
# result = await Runner.run( # .run은 실행되기까지 나오는게 없음
stream = Runner.run_streamed(agent, # verbose처럼 agent가 어떤 동작하는지 실시간으로 업데이트해 줌
                                   "Hello how are you? what is the weather in the capital of korea")

async for event in stream.stream_events():
    if event.type == 'raw_response_event':
        continue
    elif event.type == 'agent_updated_stream_event':
        print('Agent updated to', event.new_agent.name)
    elif event.type == 'run_item_stream_event':
        if event.item.type == "tool_call_item":
            print(event.item.raw_item.to_dict()) 
        elif event.item.type == "tool_call_output_item":
            print(event.item.output)            
        elif event.item.type == "message_output_item":
            print(ItemHelpers.text_message_output(event.item))
        print('='*20)
        
# itemhelpers는 item에서 text message output을 추출하는 helper function
# 그래서 결과 메세지가 매우 깔끔하게 나옴
# itemherpers 없이 실행하면 tool call item, tool call output item, message output item이 모두 raw하게 나옴(상당히 길게 더럽게 나옴)        
# 이런 과정으로 agent가 어떤 tool을 호출했고, tool이 어떤 output을 줬는지, agent가 우리에게 어떤 message를 줬는지 실시간으로 알 수 있음
'''
tool_call_item      ====> agent가 tool 호출했다는 의미
====================
tool_call_output_item ==> tool이 output 줬다는 의미
====================
message_output_item ====> agent가 우리에게 message를 줬다는 의미
====================
'''

Agent updated to Assistant Agent
{'arguments': '{"city":"Seoul"}', 'call_id': 'call_0JiMzFToBGN3DnhG05vZXcMS', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0a22cc52e3268f320069d13f44fdd48196a56c2b8d27347693', 'status': 'completed'}
Seoul
30 degrees
Hello! I’m doing well, thank you for asking. The weather in Seoul, the capital of South Korea, is currently 30 degrees (presumably Celsius). If you need more details or a forecast, just let me know!


'\ntool_call_item      ====> agent가 tool 호출했다는 의미\n====================\ntool_call_output_item ==> tool이 output 줬다는 의미\n====================\nmessage_output_item ====> agent가 우리에게 message를 줬다는 의미\n====================\n'

In [2]:
# 1:1 대응으로 결과가 깔끔하게 출력되게 만들기
# raw response item 활용
from agents import Agent, Runner, function_tool, ItemHelpers

# @function_tool
# 일반 함수를 Agent가 사용할 수 있는 도구(tool)로 만들어주는 데코레이터임
@function_tool
def get_weather(city: str):
    """get weather by city"""
    print(f"[Tool Execution] get_weather called with city: {city}")
    return "30 degrees"

# Agent
# AI 에이전트 생성: 이름, 역할(instructions), 사용 가능한 도구(tools)를 정의함
agent = Agent(
    name="Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions",
    tools=[get_weather]
)

# Runner.run_streamed
# 에이전트와 사용자 입력을 받아 실행하고, 결과를 한 번에 받지 않고 스트림(stream) 형태로 가져옴
stream = Runner.run_streamed(agent, "Hello how are you? what is the weather in the capital of korea")

message = ""
args = ""

'''
이벤트 종류별 의미
====================
raw_response_event                    ====> 모델이 보내는 원시 응답 데이터를 실시간으로 수신한다는 의미
====================
response.output_text.delta           ====> 모델이 생성한 텍스트 조각이 도착했다는 의미 (채팅 텍스트)
====================
response.function_call_arguments.delta====> 모델이 도구 호출을 위한 인자(JSON 조각)를 만들고 있다는 의미
====================
response.completed                    ====> 응답 스트리밍이 완료되었다는 의미
====================
'''

# stream_events()로 비동기 이벤트 하나씩 처리
async for event in stream.stream_events():
    if event.type == 'raw_response_event':
        event_type = event.data.type
        
        # 텍스트 조각을 모아서 출력함
        if event_type == 'response.output_text.delta':
            message += event.data.delta
            print(f"Message chunk: {message}")
            
        # 함수 호출용 인자 조각을 모아서 출력함
        elif event_type == "response.function_call_arguments.delta":
            args += event.data.delta
            print(f"Args chunk: {args}")
            
        # 응답이 완료되면 초기화함
        elif event_type == 'response.completed':
            message = ''
            args = ''
            print("--- Response Completed ---")


{"
{"city
{"city":"
{"city":"Se
{"city":"Seoul
{"city":"Seoul"}
Seoul


In [ ]:
# [추가 예시] 2번째 셀과의 차이점:
# 2번째 셀은 에이전트가 어떤 이벤트(텍스트 조각, 함수 호출)를 던지는지 내부 동작을 확인하기 위한 '디버깅/학습용' 코드라면,
# 3번째 셀은 실제 서비스(예: ChatGPT)처럼 사용자 화면에 타이핑되듯 깔끔하게 출력만 해주는 'UI 구현용' 코드입니다.
import sys

stream_example = Runner.run_streamed(
    agent, 
    "Can you explain what an AI agent is in 2 short sentences?"
)

print("Agent Response: ", end="")

async for event in stream_example.stream_events():
    if event.type == 'raw_response_event':
        event_type = event.data.type
        
        # 텍스트가 도착할 때마다 줄바꿈 없이 바로바로 화면에 뿌려줌
        # print(end="")와 sys.stdout.flush()를 이용해 스트리밍 효과를 냄
        if event_type == 'response.output_text.delta':
            print(event.data.delta, end="")
            sys.stdout.flush()
            
        elif event_type == 'response.completed':
            print("\n\n[답변 완료]")
